# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sami-shah-210504/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking (scoring).** My lane (Lane 2, Refresh / Content Opportunity Scoring) asks "which pages should an editor review first?" Every page gets a score and the pages are sorted by it. An editor can only review a limited number of pages, so the order matters more than a yes/no label.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess, pandas as pd, numpy as np
CSV = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(CSV):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
raw = pd.read_csv(CSV)
print("Raw rows:", len(raw))

Raw rows: 30000


## 2. Target or proxy

**Proxy target:** `is_declining_label = (trend_direction == "down")`. This label comes from a defined bucket in the current window, not from an observed future outcome, so it is a beginner proxy. A stronger target on the warehouse would use the prior 90 days of features to predict decline over the next 30 days. `trend_direction` and `trend_pct` are never features, because the label is built from them.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)]
df = df.drop_duplicates("content_id").reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("Pages in my slice:", len(df), "| clients:", df["client_id"].nunique())
print("Base rate declining:", round(df["is_declining_label"].mean(), 3))
print(df["is_declining_label"].value_counts())


Pages in my slice: 30000 | clients: 32
Base rate declining: 0.542
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

**Metric: Precision@50**, chosen before modelling: of the 50 pages ranked first, what share are labelled declining? A ranking only counts if it beats the base rate shown above. I will also report Precision@20. For validation I will hold out whole clients (grouped by `client_id`), because pages from one client may share patterns a model could memorise.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
banned = {"trend_direction", "trend_pct", "content_id", "client_id", "is_declining_label"}
print("Banned columns in features:", banned & set(features) or "none")

hand = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)) * df["impressions_90d"]
y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand-rule Precision@{k}:", round(precision_at_k(hand, y, k), 3))


Banned columns in features: none
Hand-rule Precision@20: 0.9
Hand-rule Precision@50: 0.68


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (a page), with its trailing-90-day metrics.** My slice keeps pages with some impressions and at least 90 days of age, deduplicated by `content_id`, and has one row per page.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
        "avg_position", "ctr", "word_count", "trend_direction", "is_declining_label"]
df[cols].head(10)

,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,3080.0,down,1
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,NaN,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,3807.0,down,1
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,NaN,down,1


## 5. Why ML beats a fixed rule here

A hand rule (stale and visible) is easy to read, but it covers very few pages. The code below shows how many pages it scores above zero, and every other page is tied at zero, so the rest of the ranking is arbitrary. A learned ranking can weigh several signals at once and score every page, and I have not yet shown that it does this better; later weeks test that on unseen clients. The action it supports: an editor reviews the top pages and chooses refresh, protect or monitor. This is decision support, not proof that a refresh causes recovery.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
n_scored = int((hand > 0).sum())
print("Pages the hand rule scores above zero:", n_scored, "of", len(df))
print("Declining rate among them:", round(y[hand.values > 0].mean(), 3))

Pages the hand rule scores above zero: 17 of 30000
Declining rate among them: 0.941


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.